In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
from cellwhisperer.config import get_path

matplotlib.style.use(get_path(["plot_style"]))
adata = sc.read_h5ad(snakemake.input[0], backed="r")


In [ ]:
# Drop 57 observation for which the date is missing
adata.obs["series_submission_date"] = pd.to_datetime(adata.obs["series_submission_date"])
adata.obsm["X_umap"] = adata.obsm["X_cellwhisperer_umap"]  # need correct name for plotting

In [ ]:
adata.obs["cluster_label"]

## Plot UMAP with labeled clusters

In [ ]:
# doudble check everything is set correctly
sc.set_figure_params(vector_friendly=True, dpi_save=400, scanpy=False)

print("Current font sizes:")
print(f"Font size: {plt.rcParams['font.size']}")
print(f"Title font size: {plt.rcParams['axes.titlesize']}")
print(f"Axis label font size: {plt.rcParams['axes.labelsize']}")
print(f"X-tick label font size: {plt.rcParams['xtick.labelsize']}")
print(f"Y-tick label font size: {plt.rcParams['ytick.labelsize']}")
print(f"Legend font size: {plt.rcParams['legend.fontsize']}")
plt.rcParams["font.family"]

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
# plot umap

sc.pl.umap(adata, color=["leiden"], legend_loc="on data", ax=ax, 
           palette=list(matplotlib.colors.CSS4_COLORS.values()), size=0.5,
           legend_fontweight="normal", 
           legend_fontsize=6,
          )
fig.savefig(snakemake.output.cluster_labeled)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
# plot umap
sc.pl.umap(adata, color=["cluster_label"], legend_loc="on data", ax=ax, 
           palette=list(matplotlib.colors.CSS4_COLORS.values()), size=2,
           legend_fontweight="normal", 
           legend_fontsize=6,
          )

In [ ]:
# 57 are incorrect. set them to mean (otherwise we need to copy the dataset, which sucks)

adata.obs.loc[adata.obs["series_submission_date"].isna(), "series_submission_date"] = adata.obs["series_submission_date"].mean()

In [ ]:
adata.obs["series_submission_date"].max(), adata.obs["series_submission_date"].min(), adata.obs["series_submission_date"].dtype

In [ ]:
import matplotlib.dates as mdates
from matplotlib.colors import Normalize

# Assuming adata.obs["series_submission_date"] is your datetime series
date_series = adata.obs["series_submission_date"]

# Normalize the dates to a range between 0 and 1
date_min = date_series.min()
date_max = date_series.max()
norm = Normalize(vmin=date_min.toordinal(), vmax=date_max.toordinal())


In [ ]:
# Choose a colormap
cmap = plt.cm.get_cmap('viridis')


# Apply the colormap to the normalized dates
# from cmcrameri import cm
# colors = cm.batlow(norm(date_series.apply(lambda x: x.toordinal())))


In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
# plot umap
sc.pl.umap(adata,  color=["series_submission_date"], legend_loc="on data", ax=ax, sort_order=False, size=0.5, 
           cmap=cmap)

# ax.set(title=...)

fig.savefig(snakemake.output.submission_date_labeled)

In [ ]:
leiden_to_label = adata.obs[["leiden", "cluster_label"]].drop_duplicates().set_index("leiden")["cluster_label"]
time_wise_leiden = adata.obs.groupby("leiden")["series_submission_date"].mean().sort_values()

adata.obs["extreme_labels"] = adata.obs["leiden"].apply(lambda cluster_id: leiden_to_label.loc[cluster_id] if cluster_id in time_wise_leiden.iloc[:10].index or cluster_id in time_wise_leiden.iloc[-10:].index else '')
adata.strings_to_categoricals()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
# plot umap


sc.pl.umap(adata, color=["extreme_labels"], legend_loc="on data", ax=ax, 
           palette=list(matplotlib.colors.CSS4_COLORS.values()), size=0.5,
           legend_fontweight="normal", 
           legend_fontsize=6,
          )


In [ ]:


xx= adata.obs.groupby("cluster_label")["series_submission_date"].mean().sort_values()

In [ ]:
xx.iloc[:10]

In [ ]:
xx.iloc[-10:]